# 👁️ Modul 05 Hands-on: Applied Computer Vision & Transfer Learning

Di hands-on ini, Anda akan mempraktikkan alur kerja Computer Vision modern:
1. **Melihat Konvolusi Bekerja**: Menginspeksi cara filter/kernel mengekstrak garis tepi (*edges*).
2. **Data Augmentation Pipeline**: Membangun transformasi data visual yang tangguh.
3. **The Transfer Learning Recipe**: Mengambil pre-trained ResNet18, membekukan backbone, dan mengganti classifier head.
4. **Training & Inference**: Melatih model pada dataset gambar dan menguji prediksinya secara visual.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"🚀 Menggunakan akselerasi: {device}")

## 1. Membedah Operasi Konvolusi: Filter Pendeteksi Garis Tepi (Sobel Filter)
Mari kita buat layer `nn.Conv2d` dengan filter manual untuk melihat bagaimana konvolusi mendeteksi batas vertikal objek secara matematis.

In [ ]:
# Buat gambar sintetis sederhana (persegi putih di latar belakang hitam)
img_synthetic = torch.zeros(1, 1, 32, 32)
img_synthetic[:, :, 8:24, 8:24] = 1.0

# Definisikan filter Sobel vertikal 3x3
sobel_kernel = torch.tensor([
    [-1., 0., 1.],
    [-2., 0., 2.],
    [-1., 0., 1.]
]).unsqueeze(0).unsqueeze(0)  # Shape: [1, 1, 3, 3]

conv_layer = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, padding=1, bias=False)
with torch.no_grad():
    conv_layer.weight = nn.Parameter(sobel_kernel)

feature_map = conv_layer(img_synthetic)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_synthetic.squeeze().numpy(), cmap="gray")
axes[0].set_title("Input Image Asli")
axes[1].imshow(feature_map.squeeze().detach().numpy(), cmap="gray")
axes[1].set_title("Feature Map: Tepi Vertikal Terdeteksi!")
plt.tight_layout()
plt.show()

## 2. Pipeline Data Augmentation Standar Industri
Untuk mencegah model menghafal posisi piksel, kita lakukan augmentasi:
* `RandomHorizontalFlip` (membalik kiri-kanan)
* `RandomRotation` (memutar sedikit)
* `Normalize` menggunakan statistik ImageNet: `mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Gunakan CIFAR-10 sebagai benchmark visual
full_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
full_val = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=val_transform)

# Ambil subset 2.000 sampel untuk demo training cepat di laptop
train_subset = Subset(full_train, range(2000))
val_subset = Subset(full_val, range(500))

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False)

classes = full_train.classes
print(f"Kelas target CIFAR-10: {classes}")

## 3. The Transfer Learning Recipe: Pretrained ResNet18
1. Load ResNet18 dengan bobot default ImageNet.
2. Bekukan seluruh layer backbone agar fitur dasarnya tidak terhapus.
3. Ganti layer `model.fc` dengan linear layer baru sesuai jumlah kelas kita (10 kelas).

In [ ]:
# 1. Load model pre-trained
weights = ResNet18_Weights.DEFAULT
model_resnet = resnet18(weights=weights)

# 2. Bekukan semua layer backbone
for param in model_resnet.parameters():
    param.requires_grad = False

# 3. Ganti classifier head terakhir
num_ftrs = model_resnet.fc.in_features  # 512 pada ResNet18
model_resnet.fc = nn.Sequential(
    nn.Linear(num_ftrs, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, len(classes))
)

model_resnet = model_resnet.to(device)

# Pastikan hanya layer classifier baru yang memerlukan gradien
trainable_params = sum(p.numel() for p in model_resnet.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model_resnet.parameters())
print(f"Total Parameter: {total_params:,}")
print(f"Parameter yang dilatih (Trainable): {trainable_params:,} (Hanya ~{trainable_params/total_params*100:.1f}%! Sangat cepat!)")

## 4. Training Classifier Head (Transfer Learning in Action)

In [ ]:
criterion = nn.CrossEntropyLoss()
# Hanya optimalkan parameter yang tidak dibekukan (model_resnet.fc.parameters())
optimizer = optim.AdamW(model_resnet.fc.parameters(), lr=0.003, weight_decay=1e-3)

print("🚀 Memulai Training Transfer Learning (3 Epochs)...")
for epoch in range(1, 4):
    model_resnet.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_resnet(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * len(labels)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += len(labels)
        
    train_acc = correct / total * 100
    
    # Evaluasi cepat
    model_resnet.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model_resnet(imgs)
            val_correct += (out.argmax(dim=1) == labels).sum().item()
            val_total += len(labels)
            
    val_acc = val_correct / val_total * 100
    print(f"Epoch {epoch}/3 | Train Loss: {running_loss/total:.4f} | Train Acc: {train_acc:.1f}% | Val Acc: {val_acc:.1f}%")

## 5. Visualisasi Hasil Prediksi
Mari kita tampilkan beberapa sampel gambar validasi bersama prediksi model.

In [ ]:
model_resnet.eval()
sample_imgs, sample_labels = next(iter(val_loader))
with torch.no_grad():
    preds = model_resnet(sample_imgs.to(device)).argmax(dim=1).cpu()

# Denormalisasi untuk visualisasi matplotlib
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    img_display = sample_imgs[i].permute(1, 2, 0).numpy()
    img_display = std * img_display + mean
    img_display = np.clip(img_display, 0, 1)
    
    true_label = classes[sample_labels[i]]
    pred_label = classes[preds[i]]
    color = "green" if true_label == pred_label else "red"
    
    ax.imshow(img_display)
    ax.set_title(f"Pred: {pred_label}\nTrue: {true_label}", color=color, fontweight="bold", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()